In [1]:
import numpy as np
import pandas as pd
import glob
from define_model import *
from my_helpers import *
import pprint as p
from sklearn.model_selection import train_test_split
from itertools import zip_longest

In [6]:
drive = 0
data_dir = "/content/drive/MyDrive/chiliSummerProject/data/" if drive==1 else "../data/"
model_path = "/content/drive/MyDrive/chiliSummerProject/models/" if drive==1 else "../models/"
identifier_eth = "TouchPose_crossGestures"
data_path_eth = data_dir + "eth_data/" 
data_path_chili= data_dir + "processed/" 
#set random seeds
np.random.seed(42)
tf.random.set_seed(42)

In [7]:
model = load_model('../models/chili_200_epochs_lr0.001_decay5e-6.h5',compile=False)

In [8]:
files = glob.glob(data_path_chili+'*.pkl')
chili_data = pd.concat([pd.read_pickle(fp) for fp in files], ignore_index=True)
chili_data['skeleton']= chili_data['skeleton'].apply(lambda x : x.astype(np.float32))
max_skel = np.max(chili_data['skeleton'].apply(lambda x : np.max(x)))
max_cap = np.max(chili_data['cap_img'].apply(lambda x : np.max(x)))
#chili_data['skeleton']= chili_data['skeleton'].apply(lambda x : x/max_skel)
chili_data['cap_img']= chili_data['cap_img'].apply(lambda x : x/max_cap)
print(chili_data.Timestamp.is_unique)
#chili_data.set_index('Timestamp', inplace=True)
#hili_data.sort_index(inplace=True)
chili_data.head()

True


,Timestamp,skeleton,ScanSizeX,ScanSizeY,trial,gesture,subject,day,cap_img
0,1.680093e+12,"[[-270.0, 288.0, 68.0], [-242.0, 276.0, 55.0],...",72,41,1,UP,32,4,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
1,1.680093e+12,"[[-270.0, 288.0, 68.0], [-243.0, 276.0, 54.0],...",72,41,1,UP,32,4,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
2,1.680093e+12,"[[-270.0, 288.0, 67.0], [-243.0, 276.0, 54.0],...",72,41,1,UP,32,4,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
3,1.680093e+12,"[[-270.0, 288.0, 67.0], [-243.0, 276.0, 54.0],...",72,41,1,UP,32,4,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
4,1.680093e+12,"[[-270.0, 288.0, 67.0], [-242.0, 276.0, 54.0],...",72,41,1,UP,32,4,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."


In [9]:
train, test_valid = train_test_split(chili_data, test_size=0.2, random_state=42, shuffle=True, stratify = chili_data['gesture'])


In [11]:
x,y = data_load(test_valid)

In [12]:
y_pred=model.predict(x)

2023-07-28 15:53:19.283453: I tensorflow/stream_executor/cuda/cuda_dnn.cc:368] Loaded cuDNN version 8204


In [23]:
def auc_pck2(y_true, y_pred):
    threshold=20
    # Calculate the Euclidean distance between predicted and true joint coordinates
    distances = tf.sqrt(tf.reduce_sum(tf.square(y_true - y_pred), axis=-1))

    # Calculate the percentage of correct keypoints for each threshold
    correct_keypoints = tf.cast(distances < threshold, tf.float32)
    pck = tf.reduce_mean(correct_keypoints) * 100.0
    return pck


def auc_pck(y_true, y_pred):
    thresholds = np.arange(20, 51, 5)  # Thresholds from 20 mm to 50 mm
    pck_values = []

    # Calculate the Euclidean distance between predicted and true joint coordinates
    distances = tf.sqrt(tf.reduce_sum(tf.square(y_true - y_pred), axis=-1))

    # Calculate the percentage of correct keypoints for each threshold
    for threshold in thresholds:
        correct_keypoints = tf.cast(distances < threshold, tf.float32)
        pck = tf.reduce_mean(correct_keypoints) * 100.0
        pck_values.append(pck)

    return tf.convert_to_tensor(pck_values, dtype=tf.float32)

print(auc_pck(y,y_pred))
auc_pck2(y,y_pred)

tf.Tensor([58.14437 80.04993 90.22259 94.05034 95.88101 97.02518 97.75327], shape=(7,), dtype=float32)


<tf.Tensor: shape=(), dtype=float32, numpy=58.14437>